# Model Pilot Test Notebook

This notebook loads the prepared workout and Fitbit data plus the saved PR prediction models so you can run a quick pilot evaluation on representative rows.

In [ ]:
## 1. Import Required Libraries

import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

print('Libraries imported successfully')

In [ ]:
## 2. Configure Model Path and Runtime Settings

root = Path.cwd().resolve().parent if 'notebooks' in str(Path.cwd()) else Path.cwd().resolve()
data_dir = root / 'data'
models_dir = root / 'models'

model_a_path = models_dir / 'model_A_workout.joblib'
model_b_path = models_dir / 'model_B_recovery.joblib'
model_c_path = models_dir / 'model_C_stacked.joblib'
metrics_path = models_dir / 'metrics_summary.json'

print(root)
print(model_a_path.exists(), model_b_path.exists(), model_c_path.exists())

In [ ]:
## 3. Load the Model and Data

processed = pd.read_csv(data_dir / 'processed_merged.csv', parse_dates=['date'])
with open(metrics_path, 'r', encoding='utf-8') as fh:
    metrics = json.load(fh)

model_a = joblib.load(model_a_path)
model_b = joblib.load(model_b_path)
model_c = joblib.load(model_c_path)

processed.head()

In [ ]:
## 4. Prepare Sample Inputs for Test Piloting

# Use a small representative sample from the processed data
sample = processed.dropna(subset=['PR_next_session']).sample(n=min(100, len(processed)), random_state=42).copy()

# Keep only columns used by the saved models
model_a_cols = [c for c in ['relative_strength', 'rolling_best_prev', 'best_est_1RM', 'pr_gap', 'distance_to_personal_best', 'total_volume', 'weekly_volume', 'volume_28d_avg', 'volume_56d_avg', 'volume_ratio', 'weekly_volume_z', 'total_sets', 'total_reps', 'avg_weight', 'max_weight', 'days_since_last_pr', 'sessions_since_last_pr', 'pr_freq_90d', 'training_age_sessions', 'training_age_days'] if c in sample.columns]
model_b_cols = [c for c in ['sleep_minutes', 'sleep_7d_avg', 'sleep_dev_from_14d', 'resting_hr', 'hr_7d_avg', 'hr_baseline_z', 'weekly_volume', 'days_since_last_workout', 'training_age_sessions'] if c in sample.columns]

X_a = sample[model_a_cols].fillna(0)
X_b = sample[model_b_cols].fillna(0)

sample[['date', 'exercise_title', 'PR_next_session']].head()

In [ ]:
## 5. Run the Model on the Sample Inputs

sample['pr_prob_workout'] = model_a.predict_proba(X_a)[:, 1]
sample['pr_prob_recovery'] = model_b.predict_proba(X_b)[:, 1]
sample['pr_prob_stacked'] = model_c.predict_proba(
    sample[['pr_prob_workout', 'pr_prob_recovery', 'relative_strength', 'pr_gap', 'days_since_last_pr', 'training_age_sessions']].fillna(0)
)[:, 1]

sample[['exercise_title', 'pr_prob_workout', 'pr_prob_recovery', 'pr_prob_stacked', 'PR_next_session']].head(10)

In [ ]:
## 6. Inspect Model Outputs and Metrics

print('Saved metrics summary:')
print(json.dumps(metrics, indent=2))

sample['prediction_bucket'] = (sample['pr_prob_stacked'] >= 0.5).astype(int)
print('\nConfusion-style breakdown:')
print(pd.crosstab(sample['prediction_bucket'], sample['PR_next_session']))

sample[['exercise_title', 'pr_prob_stacked', 'PR_next_session']].sort_values('pr_prob_stacked', ascending=False).head(10)

In [ ]:
## 7. Save Results and Log the Run

out_dir = root / 'reports' / 'pilot_runs'
out_dir.mkdir(parents=True, exist_ok=True)

sample.to_csv(out_dir / 'pilot_predictions.csv', index=False)
with open(out_dir / 'pilot_run_metadata.json', 'w', encoding='utf-8') as fh:
    json.dump({
        'timestamp': pd.Timestamp.utcnow().isoformat(),
        'sample_size': len(sample),
        'models': ['model_A_workout', 'model_B_recovery', 'model_C_stacked'],
        'metrics': metrics,
    }, fh, indent=2)

print(f'Saved pilot outputs to {out_dir}')